## **2. Feature Engineering**

### Token Level Features
&emsp;a) Imports & Load Data

In [2]:
import pandas as pd
import numpy as np
from transformers import MarianTokenizer, MarianMTModel

/Users/nathanielparkes/Desktop/Projects/Flashcards_Proj/retention_flashcards/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('data/fr_en_cleaned_train.csv')
df.head(5)

,user_id,sent_id,token,POS,Morpho-Syntactic_Features,Dependency-Relation,Dependancy-Head,time,p_recall,sentence
0,YjS/mQOx,8XTyQUAl01,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,det,2,14.0,0,Le garçon
1,YjS/mQOx,8XTyQUAl01,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,ROOT,0,14.0,0,Le garçon
2,YjS/mQOx,8XTyQUAl02,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,nsubj,4,14.0,0,Je suis une femme
3,YjS/mQOx,8XTyQUAl02,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,cop,4,14.0,0,Je suis une femme
4,YjS/mQOx,8XTyQUAl02,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,det,4,14.0,0,Je suis une femme


In [4]:
tokens = df[['token', 'POS', 'Morpho-Syntactic_Features', 'sentence']]
tokens = tokens.drop_duplicates()
tokens.head(10)
print(tokens['token'].is_unique)

duplicate_rows = tokens[tokens.duplicated(subset=['token'], keep=False)]
print(duplicate_rows[duplicate_rows['token'] == 'suis'])
# i am preserving dup tokens because they have different POS and morpho-syntactic features which could be important for the model to learn. For example "suis" can be a verb (first person singular of "être") or a noun (plural of "sui" which is a type of plant). Removing duplicates would result in loss of information that could be relevant for the model to learn the correct associations between tokens, their POS tags, and their morpho-syntactic features.

False
       token   POS                          Morpho-Syntactic_Features  \
3       suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
9       suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
13      suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
16      suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
158     suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
...      ...   ...                                                ...   
706231  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
706249  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
738926  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
740263  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
764653  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   

                                 sentence  
3                       Je suis une femme  
9                       Je su

&emsp;b) Supplment Token Level Data
 * Specifically, I am using the Lexique data base to add information about syllable count, general word frequency to give allusion to how complex or uncommon the word is

In [5]:
lexique = pd.read_csv('data/OpenLexicon.tsv', sep='\t')
lexique = lexique.rename(columns={'ortho': 'token', 'Lexique4__Lemme' : 'lemma','Lexique4__SyllNb' : 'syllable_count'})
lexique.head(5)

,token,lemma,Lexique4__Cgram,Lexique4__CgramOrtho,Lexique4__FreqOrtho,syllable_count
0,a,a,NOM,"NOM,AUX,VER",11684.443,1
1,a,avoir,VER,"NOM,AUX,VER",11684.443,1
2,a,avoir,AUX,"NOM,AUX,VER",11684.443,1
3,a capella,a capella,ADV,ADV,0.244,4
4,a cappella,a cappella,ADV,ADV,0.241,4


 * The frequency column is how many times a word occurs out of a million instances. This is heavily skewed with a range of 11000+ to smaller that 0.3. For this reason I have decided to normalize the with column with a log transformation.

In [6]:
lexique['ortho_freq'] = np.log1p(lexique['Lexique4__FreqOrtho'])
tokens = pd.merge(
    tokens.assign(token_lower=tokens['token'].str.lower()),
    lexique[['token', 'syllable_count', 'ortho_freq']].rename(columns={'token': 'token_lower'}),
    on='token_lower',
    how='left'
).drop(columns='token_lower').drop_duplicates()

tokens.head(10)



,token,POS,Morpho-Syntactic_Features,sentence,syllable_count,ortho_freq
0,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,Le garçon,1.0,9.713540
3,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,Le garçon,2.0,5.244537
4,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,Je suis une femme,1.0,10.163428
6,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,Je suis une femme,1.0,8.330842
10,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,Je suis une femme,1.0,9.039167
14,femme,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,Je suis une femme,1.0,6.585643
15,La,DET,Definite=Def|Gender=Fem|Number=Sing|fPOS=DET++,La fille,1.0,9.613319
19,fille,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,La fille,1.0,6.447431
20,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,Je suis un garçon,1.0,10.163428
22,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,Je suis un garçon,1.0,8.330842


 * There is a lot of interesting linguistic information that may have an effect on learner retention if we can **identify the lemma of the token**. Second language aquisition research suggests that learners of a language have a tendency to retain words that are congugates or closer relatives to one in their native tongue. This makes sense and I want to take advantage of this by engineering a feature column that describes the **edit distance** from the french word to english. As the lessons used in this data set are from an English based course I continue under the assumption users are more familiar with English and can use this knowledge to make inferences on French words. Although, I may consider reducing down to countries where this is particularly plausible (US, CA, etc.)

In [11]:
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-fr-en")

sentences = tokens['sentence'].unique().tolist()
fr_en_sentences = {}
for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt") # tokenize and convert to PyTorch tensors
    outputs = model.generate(**inputs) # "**" syntax unpacks inputs dict and output is tensor of english translation token ids
    translation = tokenizer.decode(outputs, skip_special_tokens=True)[0] # decode token ids to get english translation string
    fr_en_sentences[sentence] = translation

/Users/nathanielparkes/Desktop/Projects/Flashcards_Proj/retention_flashcards/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 256/256 [00:00<00:00, 13867.97it/s]


In [8]:
for fr, en in fr_en_sentences.items():
    if en[-1] == '.':
        fr_en_sentences[fr] = en[:-1]
    
for i in range(10):
    fr, en = list(fr_en_sentences.items())[i]
    print(f"French: {fr}\nEnglish: {en}\n")

French: Le garçon
English: The boy

French: Je suis une femme
English: I'm a woman

French: La fille
English: The girl

French: Je suis un garçon
English: I'm a boy

French: Je suis riche
English: I'm rich

French: Je suis rouge
English: I'm red

French: L' homme
English: Man

French: L' homme est riche
English: The man is rich

French: Une pomme
English: Apple

French: Tu es un garçon
English: You're a boy



In [9]:
def alignment_input(sents):
    file = "data/alignment_input.txt"

    with open(file, 'w', encoding='utf-8') as f:
        for src, tgt in sents.items():
            sentence = f"{src} ||| {tgt}\n"
            f.write(sentence)
        f.close()
    print(f"Alignment input file '{file}' created successfully.")

alignment_input(fr_en_sentences)

Alignment input file 'data/alignment_input.txt' created successfully.


In [19]:
import subprocess

subprocess.run([
    "awesome-align",
    "--output_file=data/alignments.txt",
    "--model_name_or_path=bert-base-multilingual-cased",
    "--data_file=data/alignment_input.txt",
    "--extraction", "softmax",
    "--batch_size", "32",
    "--output_word_file=data/aligned_word_pairs.txt",
    "--num_workers", "0"
])

Loading the dataset...


Extracting: 5148it [00:15, 333.21it/s]


CompletedProcess(args=['awesome-align', '--output_file=data/alignments.txt', '--model_name_or_path=bert-base-multilingual-cased', '--data_file=data/alignment_input.txt', '--extraction', 'softmax', '--batch_size', '32', '--output_word_file=data/aligned_word_pairs.txt', '--num_workers', '0'], returncode=0)

&emsp;d) Map word-level alignments onto tokens
 * `aligned_word_pairs.txt` was written line-for-line from `fr_en_sentences` (the same dict that built `alignment_input.txt`), so line *i* corresponds to the *i*-th sentence in `fr_en_sentences`. I add a `sent_id` column to `tokens` keyed off that order so each token row knows which alignment line it belongs to.
 * Each line is space-separated `french<sep>english` pairs. Word alignment is sparse, so **not every token receives an English mapping** (function words and morphological forms are often dropped); those rows are left as `NaN`. This `en_aligned` column is the English target for the edit-distance cognate feature.

In [12]:
# sent_id maps each sentence to its line in aligned_word_pairs.txt
# (same order the alignment input was written from: fr_en_sentences)
sentence_order = list(fr_en_sentences.keys())
sent_id_map = {sentence: i for i, sentence in enumerate(sentence_order)}
tokens['sent_id'] = tokens['sentence'].map(sent_id_map)

# Parse aligned_word_pairs.txt into {sent_id: {french_token: english_word}}
with open('data/aligned_word_pairs.txt', encoding='utf-8') as f:
    alignment_lines = [line.rstrip('\n') for line in f]

assert len(alignment_lines) == len(sentence_order), \
    "alignment lines do not line up with the number of sentences in fr_en_sentences"

fr_en_alignments = {}
for i, line in enumerate(alignment_lines):
    pair_map = {}
    for pair in line.split():
        if '<sep>' not in pair:
            continue
        fr, en = pair.split('<sep>', 1)
        # A French token can align to more than one English word in the same
        # sentence (e.g. "mange" -> "eat" / "eats"); keep them joined with " | ".
        pair_map.setdefault(fr, [])
        if en not in pair_map[fr]:
            pair_map[fr].append(en)
    fr_en_alignments[i] = {fr: ' | '.join(ens) for fr, ens in pair_map.items()}

# Look up the aligned English word for each (sent_id, token); unaligned tokens stay NaN
tokens['en_aligned'] = tokens.apply(
    lambda row: fr_en_alignments.get(row['sent_id'], {}).get(row['token']),
    axis=1,
)

coverage = tokens['en_aligned'].notna().mean()
print(f"{coverage:.1%} of token rows received an English alignment")
tokens.head(10)

90.6% of token rows received an English alignment


,token,POS,Morpho-Syntactic_Features,sentence,syllable_count,ortho_freq,sent_id,en_aligned
0,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,Le garçon,1.0,9.713540,0,The
3,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,Le garçon,2.0,5.244537,0,boy
4,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,Je suis une femme,1.0,10.163428,1,I'm
6,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,Je suis une femme,1.0,8.330842,1,I'm
10,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,Je suis une femme,1.0,9.039167,1,a
14,femme,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,Je suis une femme,1.0,6.585643,1,woman
15,La,DET,Definite=Def|Gender=Fem|Number=Sing|fPOS=DET++,La fille,1.0,9.613319,2,The
19,fille,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,La fille,1.0,6.447431,2,girl
20,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,Je suis un garçon,1.0,10.163428,3,I'm
22,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,Je suis un garçon,1.0,8.330842,3,I'm


&emsp;e) Levenshtein ratio (French token &harr; English)
 * With each token now paired to its English alignment, I measure how "cognate-like" the pair is using the **Levenshtein ratio** from `python-Levenshtein`. The ratio is a length-normalized similarity in `[0, 1]` (`1.0` = identical strings, `0.0` = no shared characters), computed as `(len(a) + len(b) - distance) / (len(a) + len(b))` over the Levenshtein edit distance — so unlike the raw edit distance it isn't biased by word length.
 * A high ratio suggests the learner can lean on English to infer the French word. Both words are lower-cased first so casing differences don't affect the score, and when a token aligns to multiple English words I take the highest ratio (the closest match). Unaligned tokens stay `NaN`.

In [13]:
import Levenshtein

# Levenshtein.ratio: length-normalized similarity in [0, 1] (1.0 = identical)
def levenshtein_ratio(token, en_aligned):
    if pd.isna(en_aligned):
        return np.nan
    # a token may align to multiple English words ("eat | eats"); take the closest
    candidates = en_aligned.split(' | ')
    return max(Levenshtein.ratio(token.lower(), cand.lower()) for cand in candidates)

tokens['lev_ratio'] = tokens.apply(
    lambda row: levenshtein_ratio(row['token'], row['en_aligned']),
    axis=1,
)
tokens.head(10)

,token,POS,Morpho-Syntactic_Features,sentence,syllable_count,ortho_freq,sent_id,en_aligned,lev_ratio
0,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,Le garçon,1.0,9.713540,0,The,0.400000
3,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,Le garçon,2.0,5.244537,0,boy,0.222222
4,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,Je suis une femme,1.0,10.163428,1,I'm,0.000000
6,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,Je suis une femme,1.0,8.330842,1,I'm,0.285714
10,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,Je suis une femme,1.0,9.039167,1,a,0.000000
14,femme,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,Je suis une femme,1.0,6.585643,1,woman,0.200000
15,La,DET,Definite=Def|Gender=Fem|Number=Sing|fPOS=DET++,La fille,1.0,9.613319,2,The,0.000000
19,fille,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,La fille,1.0,6.447431,2,girl,0.444444
20,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,Je suis un garçon,1.0,10.163428,3,I'm,0.000000
22,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,Je suis un garçon,1.0,8.330842,3,I'm,0.285714
